# Projection-balanced merging of long-trained specialists

Held-out evaluation of deterministic direction-aware merging at 1200 training steps.

In [ ]:
!nvidia-smi
!python --version

## Repository

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path
REPO='https://github.com/LeuxLello/Dlai-model-merging.git'
BRANCH='codex/multiseed-results'
WORKDIR=Path('/kaggle/working/Dlai-model-merging')
if WORKDIR.exists(): shutil.rmtree(WORKDIR)
subprocess.check_call(['git','clone','--depth','1','--branch',BRANCH,REPO,str(WORKDIR)])
subprocess.check_call([sys.executable,'-m','pip','install','-q','-e',str(WORKDIR)])
sys.path.insert(0,str(WORKDIR/'src')); os.chdir(WORKDIR)
COMMIT=subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip()
print('Commit:',COMMIT)

## Frozen protocol

In [ ]:
import gc, itertools, json, platform
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import torch
from transformers import AutoModelForSequenceClassification
from dlai_merge.diagnostics import cosine_similarity, directional_projection, incoming_norm_ratio, sign_agreement, subtract_states
from dlai_merge.evaluation import TaskEvaluator
from dlai_merge.merging import (global_ties_merge, mean_merge, projection_balanced_merge, projection_balanced_weights, task_arithmetic, ties_merge)
from dlai_merge.training import TrainConfig, train_specialist
assert torch.cuda.is_available(), 'A CUDA GPU is required.'
GPU=torch.cuda.get_device_name(0); CAP=torch.cuda.get_device_capability(0)
torch.ones(1,device='cuda').add_(1)
TASKS=['sst2','imdb','mrpc','rte','cola','boolq']; PAIRS=list(itertools.combinations(TASKS,2))
BASE='prajjwal1/bert-mini'; SUBSET_SEED=2026; STEPS=1200
DEV_SEED=42; HELDOUT_SEEDS=[7,123]; ALL_SEEDS=[DEV_SEED,*HELDOUT_SEEDS]
MAX_TRAIN=12000; MAX_EVAL=2000; MIN_WEIGHT=0.25
TA_SCALE=0.75; TIES_DENSITY=0.2; TIES_SCALE=1.0
ARTIFACT_ROOT=Path('/kaggle/working/projection_balanced_specialists')
RESULT_ROOT=Path('/kaggle/working/projection_balanced_results')
ARTIFACT_ROOT.mkdir(parents=True,exist_ok=True); RESULT_ROOT.mkdir(parents=True,exist_ok=True)
for previous in Path('/kaggle/input').glob('**/extended_study_artifacts/budget-1200'):
    shutil.copytree(previous,ARTIFACT_ROOT,dirs_exist_ok=True); print('Imported notebook-10 checkpoints:',previous)
for previous in Path('/kaggle/input').glob('**/projection_balanced_specialists'):
    shutil.copytree(previous,ARTIFACT_ROOT,dirs_exist_ok=True); print('Imported notebook-11 checkpoints:',previous)
for previous in Path('/kaggle/input').glob('**/projection_balanced_results'):
    shutil.copytree(previous,RESULT_ROOT,dirs_exist_ok=True); print('Imported notebook-11 results:',previous)
print('GPU:',GPU,'| capability:',CAP,'| visible GPUs:',torch.cuda.device_count(),'| pairs:',len(PAIRS))

## Training and evaluation

In [ ]:
def checkpoint_dir(task,seed):
    return ARTIFACT_ROOT/task/f'seed-{seed}'

def ensure_specialist(task,seed):
    run_dir=checkpoint_dir(task,seed)
    required=[run_dir/'encoder.pt',run_dir/'head.pt',run_dir/'summary.json']
    if all(path.exists() for path in required):
        return json.loads((run_dir/'summary.json').read_text())
    return train_specialist(TrainConfig(
        task=task,base_model=BASE,output_root=str(ARTIFACT_ROOT),seed=seed,subset_seed=SUBSET_SEED,
        max_train_samples=MAX_TRAIN,max_eval_samples=MAX_EVAL,max_steps=STEPS,eval_steps=100,
        epochs=3,train_batch_size=32,eval_batch_size=64,learning_rate=2e-5))

def run_seed(seed):
    out_dir=RESULT_ROOT/f'seed-{seed}'; out_dir.mkdir(parents=True,exist_ok=True)
    names=['specialists','pair_diagnostics','directional_diagnostics','weights','merge_results']
    if (out_dir/'COMPLETE').exists() and all((out_dir/f'{name}.csv').exists() for name in names):
        print(seed,'complete; loading cached tables')
        return {name:pd.read_csv(out_dir/f'{name}.csv') for name in names}
    summaries={task:ensure_specialist(task,seed) for task in TASKS}
    enc={t:torch.load(checkpoint_dir(t,seed)/'encoder.pt',map_location='cpu',weights_only=True) for t in TASKS}
    heads={t:torch.load(checkpoint_dir(t,seed)/'head.pt',map_location='cpu',weights_only=True) for t in TASKS}
    base_model=AutoModelForSequenceClassification.from_pretrained(BASE,num_labels=2)
    base={k:v.detach().cpu().clone() for k,v in base_model.base_model.state_dict().items()}
    del base_model; gc.collect(); torch.cuda.empty_cache()
    evaluators={t:TaskEvaluator(t,heads[t],base_model=BASE,seed=seed,subset_seed=SUBSET_SEED,
        max_eval_samples=MAX_EVAL,output_root=f'/kaggle/working/projection-eval-{seed}') for t in TASKS}
    references={t:evaluators[t].evaluate(enc[t])['primary_score'] for t in TASKS}
    specialist_rows=[{'seed':seed,'steps':STEPS,'task':t,'primary_metric':summaries[t]['primary_metric'],
        'score':references[t]} for t in TASKS]
    diagnostic_rows=[]; directional_rows=[]; weight_rows=[]; result_rows=[]
    for left,right in PAIRS:
        vectors={left:subtract_states(enc[left],base),right:subtract_states(enc[right],base)}
        diagnostic_rows.append({'seed':seed,'pair':f'{left}+{right}',
            'cosine_similarity':cosine_similarity(vectors[left],vectors[right]),
            'sign_agreement':sign_agreement(vectors[left],vectors[right])})
        for source,incoming in [(left,right),(right,left)]:
            directional_rows.append({'seed':seed,'pair':f'{left}+{right}','task':source,'incoming_task':incoming,
                'directional_projection':directional_projection(vectors[source],vectors[incoming]),
                'incoming_norm_ratio':incoming_norm_ratio(vectors[source],vectors[incoming])})
        left_weight,right_weight=projection_balanced_weights(base,[enc[left],enc[right]],min_weight=MIN_WEIGHT)
        weight_rows.append({'seed':seed,'pair':f'{left}+{right}','left_task':left,'right_task':right,
            'left_weight':left_weight,'right_weight':right_weight})
        candidates={'mean':mean_merge(base,[enc[left],enc[right]]),
            'task_arithmetic':task_arithmetic(base,[enc[left],enc[right]],scale=TA_SCALE),
            'tensorwise_ties':ties_merge(base,[enc[left],enc[right]],density=TIES_DENSITY,scale=TIES_SCALE),
            'global_ties':global_ties_merge(base,[enc[left],enc[right]],density=TIES_DENSITY,scale=TIES_SCALE),
            'projection_balanced':projection_balanced_merge(base,[enc[left],enc[right]],min_weight=MIN_WEIGHT)}
        for method,state in candidates.items():
            for task in (left,right):
                score=evaluators[task].evaluate(state)['primary_score']; specialist=references[task]
                result_rows.append({'seed':seed,'steps':STEPS,'pair':f'{left}+{right}','task':task,
                    'method':method,'score':score,'specialist_score':specialist,'score_delta':score-specialist,
                    'retained_ratio':score/specialist if abs(specialist)>1e-8 else np.nan})
        pd.DataFrame(result_rows).to_csv(out_dir/'merge_results_partial.csv',index=False)
        print(seed,left,right,'done')
    tables={'specialists':pd.DataFrame(specialist_rows),'pair_diagnostics':pd.DataFrame(diagnostic_rows),
        'directional_diagnostics':pd.DataFrame(directional_rows),'weights':pd.DataFrame(weight_rows),
        'merge_results':pd.DataFrame(result_rows)}
    for name,table in tables.items(): table.to_csv(out_dir/f'{name}.csv',index=False)
    (out_dir/'COMPLETE').write_text('complete')
    del evaluators,heads,enc,base; gc.collect(); torch.cuda.empty_cache()
    return tables

In [ ]:
for seed in ALL_SEEDS:
    run_seed(seed)
print('All long-training conditions complete.')

## Integrity and development view

In [ ]:
def collect(name):
    paths=sorted(RESULT_ROOT.glob(f'seed-*/{name}.csv'))
    return pd.concat([pd.read_csv(path) for path in paths],ignore_index=True)
specialists=collect('specialists'); diagnostics=collect('pair_diagnostics')
directional=collect('directional_diagnostics'); weights=collect('weights'); results=collect('merge_results')
assert len(specialists)==len(ALL_SEEDS)*len(TASKS)
assert len(diagnostics)==len(ALL_SEEDS)*len(PAIRS)
assert len(directional)==len(ALL_SEEDS)*len(PAIRS)*2
assert len(weights)==len(ALL_SEEDS)*len(PAIRS)
assert len(results)==len(ALL_SEEDS)*len(PAIRS)*5*2
assert not results.duplicated(['seed','pair','task','method']).any()
dev=(results[results.seed==DEV_SEED].groupby('method',as_index=False)
    .agg(mean_score_delta=('score_delta','mean'),worst_score_delta=('score_delta','min')))
display(specialists[specialists.seed==DEV_SEED]); display(dev.sort_values('mean_score_delta',ascending=False))

## Held-out evaluation

In [ ]:
heldout=results[results.seed.isin(HELDOUT_SEEDS)].copy()
heldout_no_cola=heldout[~heldout.pair.str.contains('cola')].copy()
pair_heldout=(heldout.groupby(['seed','pair','method'],as_index=False)
    .agg(mean_score_delta=('score_delta','mean'),worst_score_delta=('score_delta','min')))
pair_no_cola=(heldout_no_cola.groupby(['seed','pair','method'],as_index=False)
    .agg(mean_score_delta=('score_delta','mean'),worst_score_delta=('score_delta','min')))
method_heldout=(pair_heldout.groupby('method',as_index=False)
    .agg(mean_score_delta=('mean_score_delta','mean'),sd=('mean_score_delta','std'),worst=('worst_score_delta','min')))
method_no_cola=(pair_no_cola.groupby('method',as_index=False)
    .agg(mean_score_delta=('mean_score_delta','mean'),sd=('mean_score_delta','std'),worst=('worst_score_delta','min')))
display(method_heldout.sort_values('mean_score_delta',ascending=False))
display(method_no_cola.sort_values('mean_score_delta',ascending=False))

## Paired primary comparisons

In [ ]:
def paired_comparison(pair_table,baseline,seed=2026,resamples=10000):
    new=pair_table[pair_table.method=='projection_balanced'][['seed','pair','mean_score_delta','worst_score_delta']].rename(
        columns={'mean_score_delta':'projection_balanced','worst_score_delta':'projection_worst'})
    old=pair_table[pair_table.method==baseline][['seed','pair','mean_score_delta','worst_score_delta']].rename(
        columns={'mean_score_delta':'baseline','worst_score_delta':'baseline_worst'})
    paired=new.merge(old,on=['seed','pair'],validate='one_to_one')
    paired['delta']=paired.projection_balanced-paired.baseline
    paired['worst_delta']=paired.projection_worst-paired.baseline_worst
    values=paired.delta.to_numpy(); rng=np.random.default_rng(seed)
    boot=np.array([rng.choice(values,size=len(values),replace=True).mean() for _ in range(resamples)])
    summary={'baseline':baseline,'n':len(values),'mean_delta':values.mean(),'median_delta':np.median(values),
        'win_rate':(values>0).mean(),'worst_task_delta_mean':paired.worst_delta.mean(),
        'ci_low':np.quantile(boot,.025),'ci_high':np.quantile(boot,.975)}
    return paired,pd.DataFrame([summary])
paired_tensor,summary_tensor=paired_comparison(pair_no_cola,'tensorwise_ties')
paired_global,summary_global=paired_comparison(pair_no_cola,'global_ties')
comparison_summary=pd.concat([summary_tensor,summary_global],ignore_index=True)
display(comparison_summary); display(paired_tensor.sort_values('delta'))

## Held-out directionality

In [ ]:
mean_losses=heldout_no_cola[heldout_no_cola.method=='mean'][['seed','pair','task','score_delta']]
heldout_directional=directional[directional.seed.isin(HELDOUT_SEEDS)&~directional.pair.str.contains('cola')].merge(
    mean_losses,on=['seed','pair','task'],validate='one_to_one')
heldout_directional['merge_loss']=-heldout_directional.score_delta
directional_summary=[]
for seed,group in heldout_directional.groupby('seed'):
    for feature in ['directional_projection','incoming_norm_ratio']:
        directional_summary.append({'seed':seed,'feature':feature,
            'pearson_with_loss':group[feature].corr(group.merge_loss,method='pearson'),
            'spearman_with_loss':group[feature].corr(group.merge_loss,method='spearman')})
directional_summary=pd.DataFrame(directional_summary); display(directional_summary)

## Figures and export

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(17,5))
sns.barplot(data=method_no_cola.sort_values('mean_score_delta',ascending=False),x='method',y='mean_score_delta',ax=axes[0])
axes[0].tick_params(axis='x',rotation=25); axes[0].axhline(0,color='black',lw=1); axes[0].set_title('Held-out mean score change (no CoLA)')
sns.boxplot(data=paired_tensor,x='seed',y='delta',ax=axes[1]); axes[1].axhline(0,color='black',lw=1)
axes[1].set_title('Projection-balanced minus tensor-wise TIES')
weight_plot=weights[weights.seed.isin(HELDOUT_SEEDS)&~weights.pair.str.contains('cola')]
sns.histplot(data=weight_plot,x='left_weight',bins=10,ax=axes[2]); axes[2].axvline(.5,color='black',lw=1)
axes[2].set_title('Held-out projection-balanced weights')
fig.tight_layout()
FINAL=Path('/kaggle/working/projection_balanced_long_specialists'); FINAL.mkdir(exist_ok=True)
tables={'all_specialists':specialists,'all_pair_diagnostics':diagnostics,'all_directional_diagnostics':directional,
    'all_projection_weights':weights,'all_merge_results':results,'development_summary':dev,
    'heldout_method_summary':method_heldout,'heldout_no_cola_method_summary':method_no_cola,
    'paired_vs_tensorwise':paired_tensor,'paired_vs_global':paired_global,
    'paired_comparison_summary':comparison_summary,'heldout_directional_analysis':heldout_directional,
    'heldout_directional_summary':directional_summary}
for name,table in tables.items(): table.to_csv(FINAL/f'{name}.csv',index=False)
fig.savefig(FINAL/'projection_balanced_heldout.png',dpi=180,bbox_inches='tight')
metadata={'purpose':'held-out projection-balanced merging of 1200-step specialists','commit':COMMIT,
    'base_model':BASE,'tasks':TASKS,'development_seed':DEV_SEED,'heldout_seeds':HELDOUT_SEEDS,
    'subset_seed':SUBSET_SEED,'training_steps':STEPS,'min_weight':MIN_WEIGHT,
    'primary_comparator':'tensorwise_ties','primary_scope':'held-out pairs excluding CoLA',
    'methods':{'mean':{},'task_arithmetic':{'scale':TA_SCALE},
        'tensorwise_ties':{'density':TIES_DENSITY,'scale':TIES_SCALE},
        'global_ties':{'density':TIES_DENSITY,'scale':TIES_SCALE},
        'projection_balanced':{'min_weight':MIN_WEIGHT}},
    'gpu':GPU,'visible_gpu_count':torch.cuda.device_count(),'python':platform.python_version(),
    'torch':torch.__version__,'bootstrap_resamples':10000}
(FINAL/'metadata.json').write_text(json.dumps(metadata,indent=2))
shutil.make_archive('/kaggle/working/projection_balanced_long_specialists','zip',FINAL)
print('Bundle:',FINAL)